In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned district-level data
df = pd.read_csv('/workspaces/Major_project/data/processed/district_level_data.csv')

print("Shape:", df.shape)
print("\nYears:", sorted(df['year'].unique()))
print("\nStates:", df['state_name'].nunique())
print("\nDistricts:", df['district_name'].nunique())

Shape: (5104, 51)

Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

States: 36

Districts: 800


In [4]:
# Create risk level based on severity_score
def classify_risk(score):
    if score >= df['severity_score'].quantile(0.75):
        return 'High'
    elif score >= df['severity_score'].quantile(0.25):
        return 'Medium'
    else:
        return 'Low'

df['risk_level'] = df['severity_score'].apply(classify_risk)

print("Risk distribution:")
print(df['risk_level'].value_counts())
print("\nPercentages:")
print((df['risk_level'].value_counts(normalize=True) * 100).round(2))

Risk distribution:
risk_level
Medium    2552
Low       1276
High      1276
Name: count, dtype: int64

Percentages:
risk_level
Medium    50.0
Low       25.0
High      25.0
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

# Features
feature_cols = ['year', 'state_code', 'district_code', 'total_crimes', 'severity_score']

# Target
X = df[feature_cols]
y = df['risk_level']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain distribution:\n{y_train.value_counts()}")

Training set: (4083, 5)
Test set: (1021, 5)

Train distribution:
risk_level
Medium    2041
Low       1021
High      1021
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Train
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("=== ACCURACY ===")
print(f"{accuracy_score(y_test, y_pred):.4f}")

print("\n=== F1 SCORE (weighted) ===")
print(f"{f1_score(y_test, y_pred, average='weighted'):.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

=== ACCURACY ===
1.0000

=== F1 SCORE (weighted) ===
1.0000

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       255
         Low       1.00      1.00      1.00       255
      Medium       1.00      1.00      1.00       511

    accuracy                           1.00      1021
   macro avg       1.00      1.00      1.00      1021
weighted avg       1.00      1.00      1.00      1021



In [8]:
import joblib

joblib.dump(model, '../models/risk_model.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')

print("Model saved: models/risk_model.pkl")
print("Features saved: models/feature_cols.pkl")

Model saved: models/risk_model.pkl
Features saved: models/feature_cols.pkl


In [9]:
loaded_model = joblib.load('../models/risk_model.pkl')
loaded_features = joblib.load('../models/feature_cols.pkl')

# Test prediction
sample = X_test.iloc[[0]]
pred = loaded_model.predict(sample)

print("Features:")
print(sample)
print(f"\nPredicted: {pred[0]}")
print(f"Actual: {y_test.iloc[0]}")

Features:
      year  state_code  district_code  total_crimes  severity_score
4218  2022          38            464          16.0            12.0

Predicted: Low
Actual: Low
